In [1]:
# If running locally, make sure Ollama is installed and the model is pulled.
# Skip this cell if you're on Colab or using a remote Ollama endpoint.

import subprocess, shutil, sys

if shutil.which("ollama"):
    print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
else:
    print("⚠️ Ollama not found. Install from https://ollama.com/download")
    print("   Then run:  ollama pull llama3.1")

NAME                ID              SIZE      MODIFIED     
qwen2.5vl:latest    5ced39dfa4ba    6.0 GB    24 hours ago    
qwen3.5:4b          d8b0f5e9760c    3.3 GB    25 hours ago    



In [1]:
import json
import re
from typing import Any, Dict, List, Optional

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "qwen2.5vl:latest"     # try: mistral, qwen2.5, llama3.2
TEMPERATURE     = 0

In [21]:
from pydantic import BaseModel, Field

class ClaimExtraction(BaseModel):
    """Structured output schema for claim extraction."""

    claim_amount: Optional[float] = Field(
        default=None,
        description="Total claim/billed amount in USD. Numeric only."
    )
    icd10_codes: List[str] = Field(
        default_factory=list,
        description="ICD-10 diagnosis codes (e.g., E11.9, I10)."
    )
    cpt_codes: List[str] = Field(
        default_factory=list,
        description="CPT procedure codes (5-digit, e.g., 99213, 93000)."
    )
    provider_npi: Optional[str] = Field(
        default=None,
        description="10-digit National Provider Identifier."
    )
    service_date: Optional[str] = Field(
        default=None,
        description="Date of service in MM-DD-YYYY (USA) format."
    )
    physician_signature: Optional[str] = Field(
        default=None,
        description="Signature of the attending physician."
    )
    confidence: Optional[float] = Field(
        default=None,
        description="Model confidence 0.0-1.0."
    )
    notes: Optional[str] = Field(
        default=None,
        description="Any ambiguity or missing fields worth flagging."
    )

In [19]:
SYSTEM_PROMPT = """You are a meticulous medical claims extraction agent.

Extract these fields from the provided claim context:
1. claim_amount  -> total billed/claim amount (numeric, USD)
2. icd10_codes   -> list of ICD-10 diagnosis codes (e.g., E11.9, I10, M54.5)
3. cpt_codes     -> list of CPT procedure codes (5-digit, e.g., 99213, 93000)
4. provider_npi  -> 10-digit National Provider Identifier
5. service_date  -> date of service in MM-DD-YYYY (USA)format
6. Physician signature -> Signature of the attending physician

Rules:
- Only extract what is explicitly present. Do NOT hallucinate.
- Normalize ICD-10 codes: uppercase, keep the dot (e.g., "e119" -> "E11.9").
- CPT codes must be exactly 5 digits.
- NPI must be exactly 10 digits.
- If a field is missing, return null (or an empty list for codes).
- Output MUST be valid JSON matching the schema. No markdown, no commentary.
"""

USER_PROMPT = """Extract the claim fields from the context below.

--- CONTEXT START ---
{context}
--- CONTEXT END ---

Return ONLY valid JSON with these keys:
claim_amount, icd10_codes, cpt_codes, provider_npi, service_date, physician_signature, confidence, notes
"""

claim_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human",  USER_PROMPT),
])

In [18]:
ICD10_RE = re.compile(r"^[A-Z]\d{2}(\.\d{1,4})?$")
CPT_RE   = re.compile(r"^\d{5}$")
NPI_RE   = re.compile(r"^\d{10}$")

def validate_codes(result: Dict[str, Any]) -> Dict[str, Any]:
    """Sanitize/normalize codes and drop invalid entries."""
    notes = result.get("notes") or ""

    npi = result.get("provider_npi")
    if npi and not NPI_RE.fullmatch(str(npi)):
        notes += f" | Invalid NPI format: {npi}"
        result["provider_npi"] = None

    result["cpt_codes"] = [
        c for c in result.get("cpt_codes", [])
        if isinstance(c, str) and CPT_RE.fullmatch(c)
    ]
    result["icd10_codes"] = [
        c.upper() for c in result.get("icd10_codes", [])
        if isinstance(c, str) and ICD10_RE.fullmatch(c.upper())
    ]

    if notes:
        result["notes"] = notes.strip(" |")
    return result

In [16]:
class ClaimsAgent:
    """Extracts structured claim data from unstructured context."""

    def __init__(self, model: str = OLLAMA_MODEL):
        self.llm = ChatOllama(
            base_url=OLLAMA_BASE_URL,
            model=model,
            temperature=TEMPERATURE,
            format="json",
        )
        self.parser = JsonOutputParser(pydantic_object=ClaimExtraction)
        self.chain = claim_prompt | self.llm | RunnableLambda(self._safe_parse)

    @staticmethod
    def _safe_parse(message) -> Dict[str, Any]:
        raw = message.content if hasattr(message, "content") else str(message)
        raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

        try:
            data = json.loads(raw)
        except json.JSONDecodeError as e:
            return {"error": f"Invalid JSON: {e}", "raw_output": raw}

        try:
            validated = ClaimExtraction(**data)
            return validate_codes(validated.model_dump())
        except Exception as e:
            return {"validation_error": str(e), "parsed": data}

    def extract(self, context: str) -> Dict[str, Any]:
        if not context or not context.strip():
            raise ValueError("Context is empty.")
        return self.chain.invoke({"context": context})

In [25]:
PatientID = 'PT_62350'
claim_forms = f'd:/Project/AI/Project/dataset/claim_forms/claim_{PatientID}.png'
claim_forms

'd:/Project/AI/Project/dataset/claim_forms/claim_PT_62350.png'

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")

In [ ]:
import base64
import os
import mimetypes
import json
from pathlib import Path
from typing import Union, Optional, Dict, Any, List
from groq import Groq

# Initialize once — reuses the connection
_client: Optional[Groq] = None

def _get_client() -> Groq:
    global _client
    if _client is None:
        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            raise ValueError("GROQ_API_KEY environment variable not set.")
        _client = Groq(api_key=api_key)
    return _client


def _encode_image(image_path: str) -> tuple[str, str]:
    """Returns (base64_string, mime_type)."""
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    mime_type, _ = mimetypes.guess_type(path.name)
    mime_type = mime_type or "image/jpeg"

    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return encoded, mime_type


def read_image(image: Union[str, List[str]],prompt: str = "Extract all text and key data from this image.", model: str = OLLAMA_MODEL,*,
    json_schema: Optional[Dict[str, Any]] = None,temperature: float = 1.0,max_tokens: int = 2048,stream: bool = False,) -> Union[str, Dict[str, Any]]:
    """
    Read and extract data from one or more images using Groq's vision models.

    Args:
        image:       Local file path, URL, or a list of either (max 3).
        prompt:      Instruction for the model.
        model:       Groq vision model ID.
        json_schema: Optional JSON schema — if provided, returns a dict.
        temperature: Sampling temperature.
        max_tokens:  Max completion tokens.
        stream:      Stream the response (returns generator).

    Returns:
        String, dict (if json_schema given), or generator (if stream=True).
    """
    client = _get_client()

    # Normalize to list
    images = [image] if isinstance(image, str) else image
    if not images:
        raise ValueError("At least one image is required.")
    if len(images) > 3:
        raise ValueError("Groq supports a maximum of 3 images per request.")

    # Build content blocks
    content: List[Dict[str, Any]] = [{"type": "text", "text": prompt}]

    for img in images:
        if img.startswith(("http://", "https://")):
            url = img
        else:
            b64, mime = _encode_image(img)
            url = f"data:{mime};base64,{b64}"
        content.append({
            "type": "image_url",
            "image_url": {"url": url},
        })

    # Build request kwargs
    kwargs: Dict[str, Any] = {
        "model": model,
        "messages": [{"role": "user", "content": content}],
        "temperature": temperature,
        "max_completion_tokens": max_tokens,
        "stream": stream,
    }

    if json_schema:
        kwargs["response_format"] = {"type": "json_object"}
        # Inject schema into the prompt so the model knows what to produce
        schema_hint = (
            "\n\nRespond ONLY with valid JSON matching this schema:\n"
            f"{json.dumps(json_schema, indent=2)}"
        )
        content[0]["text"] += schema_hint

    completion = client.chat.completions.create(**kwargs)

    if stream:
        return _stream_response(completion)

    raw = completion.choices[0].message.content

    if json_schema:
        try:
            return json.loads(raw)
        except json.JSONDecodeError as e:
            raise ValueError(f"Model did not return valid JSON: {raw}") from e

    return raw


def _stream_response(completion):
    """Yield text chunks from a streaming completion."""
    for chunk in completion:
        delta = chunk.choices[0].delta.content
        if delta:
            yield delta

In [ ]:
SAMPLE_CONTEXT = ""

In [27]:
SAMPLE_CONTEXT_TYPE2 = read_image(claim_forms)

In [22]:
agent = ClaimsAgent(model=OLLAMA_MODEL)
result = agent.extract(SAMPLE_CONTEXT)
print(json.dumps(result, indent=2))

{
  "validation_error": "1 validation error for ClaimExtraction\nconfidence\n  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='high', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.14/v/float_parsing",
  "parsed": {
    "claim_amount": "4095.00",
    "icd10_codes": [
      "I25.10",
      "G43.909"
    ],
    "cpt_codes": [
      "90837"
    ],
    "provider_npi": "NPI-5610293847",
    "service_date": "12-15-2025",
    "physician_signature": "Dr. Rachel Kim, MD\nDate: 12/03/2025",
    "confidence": "high",
    "notes": "The sum of the listed line items is $9,095.00, which conflicts with the displayed page total of $4,095.00."
  }
}


In [23]:
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any
import re
import json

class ClaimExtraction(BaseModel):
    """Structured output schema for claim extraction."""

    claim_amount: Optional[float] = Field(
        default=None,
        description="Total claim/billed amount in USD. Numeric only."
    )
    icd10_codes: List[str] = Field(
        default_factory=list,
        description="ICD-10 diagnosis codes (e.g., E11.9, I10)."
    )
    cpt_codes: List[str] = Field(
        default_factory=list,
        description="CPT procedure codes (5-digit, e.g., 99213, 93000)."
    )
    provider_npi: Optional[str] = Field(
        default=None,
        description="10-digit National Provider Identifier."
    )
    service_date: Optional[str] = Field(
        default=None,
        description="Date of service in MM-DD-YYYY (USA) format."
    )
    physician_signature: Optional[str] = Field(
        default=None,
        description="Signature of the attending physician."
    )
    confidence: Optional[float] = Field(
        default=None,
        description="Model confidence 0.0-1.0."
    )
    notes: Optional[str] = Field(
        default=None,
        description="Any ambiguity or missing fields worth flagging."
    )


SYSTEM_PROMPT = """You are a meticulous medical claims extraction agent.

Extract these fields from the provided claim context:
1. claim_amount  -> total billed/claim amount (numeric, USD)
2. icd10_codes   -> list of ICD-10 diagnosis codes (e.g., E11.9, I10, M54.5)
3. cpt_codes     -> list of CPT procedure codes (5-digit, e.g., 99213, 93000)
4. provider_npi  -> 10-digit National Provider Identifier
5. service_date  -> date of service in YYYY-MM-DD format
6. Physician signature -> Signature of the attending physician

Rules:
- Only extract what is explicitly present. Do NOT hallucinate.
- Normalize ICD-10 codes: uppercase, keep the dot (e.g., "e119" -> "E11.9").
- CPT codes must be exactly 5 digits.
- NPI must be exactly 10 digits.
- Dates must be normalized to YYYY-MM-DD.
- If a field is missing, return null (or an empty list for codes).
- Output MUST be valid JSON matching the schema. No markdown, no commentary.
"""

USER_PROMPT = """Extract the claim fields from the context below.

--- CONTEXT START ---
{context}
--- CONTEXT END ---

Return ONLY valid JSON with these keys:
claim_amount, icd10_codes, cpt_codes, provider_npi, service_date, physician_signature, confidence, notes
"""

# Regex validators
ICD10_RE = re.compile(r"^[A-Z]\d{2}(\.\d{1,4})?$")
CPT_RE = re.compile(r"^\d{5}$")
NPI_RE = re.compile(r"^\d{10}$")
DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")

# Fields that are always required in the output (per your schema)
REQUIRED_FIELDS = {
    "claim_amount": "claim_amount is required and cannot be null.",
    "icd10_codes": "icd10_codes is required and cannot be empty.",
    "cpt_codes": "cpt_codes is required and cannot be empty.",
    "provider_npi": "provider_npi is required and cannot be null.",
    "service_date": "service_date is required and cannot be null.",
    "physician_signature": "physician_signature is required and cannot be null.",
}


def normalize_date(raw: str) -> Optional[str]:
    """Convert various date formats to YYYY-MM-DD."""
    if not raw:
        return None
    raw = str(raw).strip()
    # Already ISO
    if DATE_RE.match(raw):
        return raw
    # MM-DD-YYYY or MM/DD/YYYY
    m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{4})$", raw)
    if m:
        mm, dd, yyyy = m.groups()
        return f"{yyyy}-{int(mm):02d}-{int(dd):02d}"
    # DD-MM-YYYY fallback
    m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})$", raw)
    if m:
        a, b, yyyy = m.groups()
        if len(yyyy) == 2:
            yyyy = "20" + yyyy
        return f"{yyyy}-{int(a):02d}-{int(b):02d}"
    return None


def validate_and_format(result: Dict[str, Any]) -> Dict[str, Any]:
    """
    Sanitize/normalize extracted fields and build the final output envelope
    with schema_valid flag and validation_errors list.
    """
    validation_errors: List[Dict[str, str]] = []
    extracted: Dict[str, Any] = {}

    # ---- claim_amount ----
    amt = result.get("claim_amount")
    if amt is None:
        extracted["claim_amount"] = None
    else:
        try:
            extracted["claim_amount"] = float(amt)
        except (TypeError, ValueError):
            extracted["claim_amount"] = None
            validation_errors.append({
                "field": "claim_amount",
                "code": "INVALID_FORMAT",
                "message": f"claim_amount must be numeric, got: {amt}",
            })

    # ---- icd10_codes ----
    icd_raw = result.get("icd10_codes") or []
    icd_clean = []
    for c in icd_raw:
        if not isinstance(c, str):
            validation_errors.append({
                "field": "icd10_codes",
                "code": "INVALID_FORMAT",
                "message": f"ICD-10 code must be a string, got: {c}",
            })
            continue
        c_up = c.upper().strip()
        if ICD10_RE.fullmatch(c_up):
            icd_clean.append(c_up)
        else:
            validation_errors.append({
                "field": "icd10_codes",
                "code": "INVALID_FORMAT",
                "message": f"Invalid ICD-10 code format: {c}",
            })
    extracted["icd10_codes"] = icd_clean

    # ---- cpt_codes ----
    cpt_raw = result.get("cpt_codes") or []
    cpt_clean = []
    for c in cpt_raw:
        if not isinstance(c, str):
            validation_errors.append({
                "field": "cpt_codes",
                "code": "INVALID_FORMAT",
                "message": f"CPT code must be a string, got: {c}",
            })
            continue
        c_str = c.strip()
        if CPT_RE.fullmatch(c_str):
            cpt_clean.append(c_str)
        else:
            validation_errors.append({
                "field": "cpt_codes",
                "code": "INVALID_FORMAT",
                "message": f"Invalid CPT code format: {c}",
            })
    extracted["cpt_codes"] = cpt_clean

    # ---- provider_npi ----
    npi = result.get("provider_npi")
    if npi is None or str(npi).strip() == "":
        extracted["provider_npi"] = None
    else:
        npi_str = str(npi).strip()
        if NPI_RE.fullmatch(npi_str):
            extracted["provider_npi"] = npi_str
        else:
            extracted["provider_npi"] = None
            validation_errors.append({
                "field": "provider_npi",
                "code": "INVALID_FORMAT",
                "message": f"Invalid NPI format (must be 10 digits): {npi}",
            })

    # ---- service_date ----
    raw_date = result.get("service_date")
    normalized = normalize_date(raw_date) if raw_date else None
    if raw_date and not normalized:
        validation_errors.append({
            "field": "service_date",
            "code": "INVALID_FORMAT",
            "message": f"Could not parse service_date: {raw_date}",
        })
    extracted["service_date"] = normalized

    # ---- Required-field checks ----
    for field, msg in REQUIRED_FIELDS.items():
        val = extracted.get(field)
        is_missing = val is None or (isinstance(val, list) and len(val) == 0)
        if is_missing:
            # Avoid duplicate INVALID_FORMAT + REQUIRED_MISSING for same field
            already = any(e["field"] == field for e in validation_errors)
            if not already:
                validation_errors.append({
                    "field": field,
                    "code": "REQUIRED_MISSING",
                    "message": msg,
                })

    schema_valid = len(validation_errors) == 0

    return {
        "extracted_fields": extracted,
        "schema_valid": schema_valid,
        "validation_errors": validation_errors,
    }


class ClaimsAgent:
    """Extracts structured claim data from unstructured context."""

    def __init__(self, model: str = OLLAMA_MODEL):
        self.llm = ChatOllama(
            base_url=OLLAMA_BASE_URL,
            model=model,
            temperature=TEMPERATURE,
            format="json",
        )
        self.parser = JsonOutputParser(pydantic_object=ClaimExtraction)
        self.chain = claim_prompt | self.llm | RunnableLambda(self._safe_parse)

    @staticmethod
    def _safe_parse(message) -> Dict[str, Any]:
        raw = message.content if hasattr(message, "content") else str(message)
        raw = (
            raw.strip()
            .removeprefix("```json")
            .removeprefix("```")
            .removesuffix("```")
            .strip()
        )

        try:
            data = json.loads(raw)
        except json.JSONDecodeError as e:
            return {
                "extracted_fields": {
                    "claim_amount": None,
                    "icd10_codes": [],
                    "cpt_codes": [],
                    "provider_npi": None,
                    "service_date": None,
                    "physician_signature": None,
                },
                "schema_valid": False,
                "validation_errors": [
                    {
                        "field": "_raw_output",
                        "code": "INVALID_JSON",
                        "message": f"Model did not return valid JSON: {e}",
                    }
                ],
            }

        try:
            validated = ClaimExtraction(**data)
            return validate_and_format(validated.model_dump())
        except Exception as e:
            return {
                "extracted_fields": data,
                "schema_valid": False,
                "validation_errors": [
                    {
                        "field": "_schema",
                        "code": "VALIDATION_ERROR",
                        "message": str(e),
                    }
                ],
            }

    def extract(self, context: str) -> Dict[str, Any]:
        if not context or not context.strip():
            raise ValueError("Context is empty.")
        return self.chain.invoke({"context": context})




In [24]:
# ---- Usage ----
agent = ClaimsAgent(model=OLLAMA_MODEL)
result = agent.extract(SAMPLE_CONTEXT)
print(json.dumps(result, indent=2))

{
  "extracted_fields": {
    "claim_amount": "4095.00",
    "icd10_codes": [
      "I25.10",
      "G43.909"
    ],
    "cpt_codes": [
      "90837"
    ],
    "provider_npi": "NPI-5610293847",
    "service_date": "12-15-2025",
    "physician_signature": "Dr. Rachel Kim, MD\nDate: 12/03/2025",
    "confidence": "high",
    "notes": "The sum of the listed line items is $9,095.00, which conflicts with the displayed page total of $4,095.00."
  },
  "schema_valid": false,
  "validation_errors": [
    {
      "field": "_schema",
      "code": "VALIDATION_ERROR",
      "message": "1 validation error for ClaimExtraction\nconfidence\n  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='high', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.14/v/float_parsing"
    }
  ]
}


In [11]:
# Cell 2
def extract_claim(
    context: str,
    model: str = OLLAMA_MODEL,
    base_url: str = "http://localhost:11434",
    temperature: float = 0.0,
    cpt_description_lookup: dict | None = None,
) -> dict:
    """
    Extract structured claim data (with CPT descriptions) from free-text context.

    Returns a dict:
        {
          "extracted_fields": {...},
          "schema_valid": bool,
          "validation_errors": [...]
        }
    """
    # ---- lazy imports (keeps notebook namespace clean) ---------------------
    import re
    import json
    from typing import Optional, List, Dict, Any, Union

    from pydantic import BaseModel, Field
    from langchain_ollama import ChatOllama
    from langchain_core.output_parsers import JsonOutputParser
    from langchain_core.runnables import RunnableLambda
    from langchain_core.prompts import ChatPromptTemplate

    # ---- schema ------------------------------------------------------------
    class CPTCode(BaseModel):
        code: str = Field(..., description="5-digit CPT code, e.g. 99213.")
        description: Optional[str] = Field(
            default=None,
            description="Plain-English procedure description, e.g. 'CBC with differential'.",
        )

    class ClaimExtraction(BaseModel):
        claim_amount: Optional[float] = Field(
            default=None, description="Total billed amount in USD."
        )
        icd10_codes: List[str] = Field(
            default_factory=list, description="ICD-10 codes."
        )
        cpt_codes: List[Union[str, CPTCode]] = Field(
            default_factory=list,
            description='CPT codes as strings or {"code","description"} objects.',
        )
        provider_npi: Optional[str] = Field(
            default=None, description="10-digit NPI."
        )
        service_date: Optional[str] = Field(
            default=None, description="Date of service, YYYY-MM-DD."
        )
        physician_signature: Optional[str] = Field(
            default=None, description="Attending physician signature."
        )
        confidence: Optional[float] = Field(
            default=None, description="Model confidence 0.0-1.0."
        )
        notes: Optional[str] = Field(
            default=None, description="Ambiguity / missing-field notes."
        )

    # ---- prompts -----------------------------------------------------------
    system_prompt = """You are a meticulous medical claims extraction agent.

        Extract these fields from the claim context:
        1. claim_amount  -> total billed/claim amount (numeric, USD)
        2. icd10_codes   -> list of ICD-10 diagnosis codes (e.g., E11.9, I10)
        3. cpt_codes     -> list of CPT procedure codes as OBJECTS:
                            {"code": "<5-digit code>", "description": "<plain-English description>"}
                            - description is the text appearing next to the code in the
                            context (e.g. "CBC with differential").
                            - If no description is present, set description to null.
                            - Do NOT invent descriptions.
                            - Acceptable source formats include:
                                "85025 - CBC with differential"
                                "CPT 43239 (Upper GI endoscopy with biopsy)"
                                "Procedure: 80053 — Comprehensive metabolic panel"
        4. provider_npi  -> 10-digit NPI
        5. service_date  -> date of service in YYYY-MM-DD
        6. physician_signature -> signature of the attending physician

        Rules:
        - Only extract what is explicitly present. Do NOT hallucinate.
        - ICD-10: uppercase, keep the dot (e.g. "e119" -> "E11.9").
        - CPT: exactly 5 digits, as strings.
        - NPI: exactly 10 digits.
        - Dates: normalize to YYYY-MM-DD.
        - Missing field -> null (or [] for lists).
        - Output MUST be valid JSON. No markdown, no commentary.
        """

    user_prompt = """Extract the claim fields from the context below.

        --- CONTEXT START ---
        {context}
        --- CONTEXT END ---

        Return ONLY valid JSON with these keys:
        claim_amount, icd10_codes, cpt_codes, provider_npi, service_date,
        physician_signature, confidence, notes

        cpt_codes MUST be a list of objects:
        [{{"code": "85025", "description": "CBC with differential"}}, ...]
        """

    # ---- regexes -----------------------------------------------------------
    ICD10_RE = re.compile(r"^[A-Z]\d{2}(\.\d{1,4})?$")
    CPT_RE   = re.compile(r"^\d{5}$")
    NPI_RE   = re.compile(r"^\d{10}$")
    DATE_RE  = re.compile(r"^\d{4}-\d{2}-\d{2}$")

    REQUIRED_FIELDS = {
        "claim_amount":        "claim_amount is required and cannot be null.",
        "icd10_codes":         "icd10_codes is required and cannot be empty.",
        "cpt_codes":           "cpt_codes is required and cannot be empty.",
        "provider_npi":        "provider_npi is required and cannot be null.",
        "service_date":        "service_date is required and cannot be null.",
        "physician_signature": "physician_signature is required and cannot be null.",
    }

    lookup: Dict[str, str] = dict(cpt_description_lookup or {})

    # ---- helpers -----------------------------------------------------------
    def normalize_date(raw):
        if not raw:
            return None
        raw = str(raw).strip()
        if DATE_RE.match(raw):
            return raw
        m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{4})$", raw)
        if m:
            mm, dd, yyyy = m.groups()
            return f"{yyyy}-{int(mm):02d}-{int(dd):02d}"
        m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})$", raw)
        if m:
            a, b, yyyy = m.groups()
            if len(yyyy) == 2:
                yyyy = "20" + yyyy
            return f"{yyyy}-{int(a):02d}-{int(b):02d}"
        return None

    def normalize_cpt_entry(c):
        if isinstance(c, str):
            return {"code": c.strip(), "description": None}
        if isinstance(c, dict):
            code = str(c.get("code", "")).strip()
            raw_desc = c.get("description")
            desc = str(raw_desc).strip() if raw_desc is not None else None
            return {"code": code, "description": desc or None}
        return None

    def validate_and_format(result):
        errors: List[Dict[str, str]] = []
        out: Dict[str, Any] = {}

        # claim_amount
        amt = result.get("claim_amount")
        if amt is None:
            out["claim_amount"] = None
        else:
            try:
                out["claim_amount"] = float(amt)
            except (TypeError, ValueError):
                out["claim_amount"] = None
                errors.append({
                    "field": "claim_amount",
                    "code": "INVALID_FORMAT",
                    "message": f"claim_amount must be numeric, got: {amt}",
                })

        # icd10_codes
        icd_clean = []
        for c in (result.get("icd10_codes") or []):
            if not isinstance(c, str):
                errors.append({
                    "field": "icd10_codes",
                    "code": "INVALID_FORMAT",
                    "message": f"ICD-10 code must be a string, got: {c}",
                })
                continue
            cu = c.upper().strip()
            if ICD10_RE.fullmatch(cu):
                icd_clean.append(cu)
            else:
                errors.append({
                    "field": "icd10_codes",
                    "code": "INVALID_FORMAT",
                    "message": f"Invalid ICD-10 code format: {c}",
                })
        out["icd10_codes"] = icd_clean

        # cpt_codes (with descriptions)
        cpt_clean = []
        for c in (result.get("cpt_codes") or []):
            entry = normalize_cpt_entry(c)
            if entry is None:
                errors.append({
                    "field": "cpt_codes",
                    "code": "INVALID_FORMAT",
                    "message": f"CPT entry must be string or object, got: {c!r}",
                })
                continue
            code, desc = entry["code"], entry["description"]
            if not CPT_RE.fullmatch(code):
                errors.append({
                    "field": "cpt_codes",
                    "code": "INVALID_FORMAT",
                    "message": f"Invalid CPT code format: {code!r} (from {c!r})",
                })
                continue
            if desc is None:
                desc = lookup.get(code)
            cpt_clean.append({"code": code, "description": desc})
        out["cpt_codes"] = cpt_clean

        # provider_npi
        npi = result.get("provider_npi")
        if npi is None or str(npi).strip() == "":
            out["provider_npi"] = None
        else:
            npi_str = str(npi).strip()
            if NPI_RE.fullmatch(npi_str):
                out["provider_npi"] = npi_str
            else:
                out["provider_npi"] = None
                errors.append({
                    "field": "provider_npi",
                    "code": "INVALID_FORMAT",
                    "message": f"Invalid NPI format (must be 10 digits): {npi}",
                })

        # service_date
        raw_date = result.get("service_date")
        norm = normalize_date(raw_date) if raw_date else None
        if raw_date and not norm:
            errors.append({
                "field": "service_date",
                "code": "INVALID_FORMAT",
                "message": f"Could not parse service_date: {raw_date}",
            })
        out["service_date"] = norm

        # physician_signature
        sig = result.get("physician_signature")
        out["physician_signature"] = (
            str(sig).strip() if sig is not None and str(sig).strip() else None
        )

        # confidence / notes
        conf = result.get("confidence")
        try:
            out["confidence"] = float(conf) if conf is not None else None
        except (TypeError, ValueError):
            out["confidence"] = None
        notes = result.get("notes")
        out["notes"] = str(notes).strip() if notes else None

        # required-field checks
        for field, msg in REQUIRED_FIELDS.items():
            val = out.get(field)
            is_missing = val is None or (isinstance(val, list) and len(val) == 0)
            if is_missing and not any(e["field"] == field for e in errors):
                errors.append({
                    "field": field,
                    "code": "REQUIRED_MISSING",
                    "message": msg,
                })

        return {
            "extracted_fields": out,
            "schema_valid": len(errors) == 0,
            "validation_errors": errors,
        }

    def safe_parse(message):
        raw = message.content if hasattr(message, "content") else str(message)
        raw = (
            raw.strip()
            .removeprefix("```json")
            .removeprefix("```")
            .removesuffix("```")
            .strip()
        )
        try:
            data = json.loads(raw)
        except json.JSONDecodeError as e:
            return {
                "extracted_fields": {
                    "claim_amount": None,
                    "icd10_codes": [],
                    "cpt_codes": [],
                    "provider_npi": None,
                    "service_date": None,
                    "physician_signature": None,
                },
                "schema_valid": False,
                "validation_errors": [{
                    "field": "_raw_output",
                    "code": "INVALID_JSON",
                    "message": f"Model did not return valid JSON: {e}",
                }],
            }
        try:
            validated = ClaimExtraction(**data)
            return validate_and_format(validated.model_dump())
        except Exception as e:
            return {
                "extracted_fields": data,
                "schema_valid": False,
                "validation_errors": [{
                    "field": "_schema",
                    "code": "VALIDATION_ERROR",
                    "message": str(e),
                }],
            }

    # ---- build chain and run ----------------------------------------------
    if not context or not context.strip():
        raise ValueError("Context is empty.")

    llm = ChatOllama(
        base_url=base_url,
        model=model,
        temperature=temperature,
        format="json",
    )
    # JsonOutputParser is kept for parity with your original code, but the
    # chain output is parsed by safe_parse for robustness.
    _ = JsonOutputParser(pydantic_object=ClaimExtraction)

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", user_prompt),
    ])
    chain = prompt | llm | RunnableLambda(safe_parse)

    return chain.invoke({"context": context})

In [ ]:
print(SAMPLE_CONTEXT)

UB-04
CMS-1450 UNIFORM INSTITUTIONAL CLAIM FORM
APPROVED OMB NO. 0938-0997 | TYPE OF BILL: 0111 – Hospital Inpatient Admit-Through-Discharge

**SECTION 1: BILLING PROVIDER NAME / ADDRESS / PHONE**
Northwestern Memorial Hospital
251 E Huron St, Chicago, IL 60611
Ph: (312) 926-2000 Fax: (312) 926-8111
NPI: NPI-5610293847

**SECTION 3b: MEDICAL RECORD NO.**
MR-PT_19116

**SECTION 4: TYPE OF BILL**
0111

**SECTION 5: FED. TAX NO.**
36-4721035

**SECTION 6: STATEMENT COVERS PERIOD**
FROM 11/30/2025 THRU 12/03/2025

**SECTION 8: PATIENT NAME (Last, First, M)**
Christopher Harris I.

**SECTION 9: PATIENT ADDRESS**
456 Oak Ave, Minneapolis, MN
55412-5133

**SECTION 10: BIRTH DATE**
04/25/1954

**SECTION 11: SEX**
M

**SECTION 12: ADMISSION DATE**
11/30/2025

**SECTION 14: ADMIT TYPE**
1 – Emergency

**SECTION 17: PATIENT DISCHARGE STATUS**
01 – Discharged to Home / Self-Care

**SECTION 18–28 CONDITION CODES**
05 Lien
39 Private Room Medically Necessary

---

**BILLING LINES:**

| 42 REV CD | 4

In [ ]:
CONTEXT_TYPE_1 = """
UB-04
CMS-1450 UNIFORM INSTITUTIONAL CLAIM FORM
APPROVED OMB NO. 0938-0997 | TYPE OF BILL: 0111 – Hospital Inpatient Admit-Through-Discharge

**SECTION 1: BILLING PROVIDER NAME / ADDRESS / PHONE**
Northwestern Memorial Hospital
251 E Huron St, Chicago, IL 60611
Ph: (312) 926-2000 Fax: (312) 926-8111
NPI: NPI-5610293847

**SECTION 3b: MEDICAL RECORD NO.**
MR-PT_19116

**SECTION 4: TYPE OF BILL**
0111

**SECTION 5: FED. TAX NO.**
36-4721035

**SECTION 6: STATEMENT COVERS PERIOD**
FROM 11/30/2025 THRU 12/03/2025

**SECTION 8: PATIENT NAME (Last, First, M)**
Christopher Harris I.

**SECTION 9: PATIENT ADDRESS**
456 Oak Ave, Minneapolis, MN
55412-5133

**SECTION 10: BIRTH DATE**
04/25/1954

**SECTION 11: SEX**
M

**SECTION 12: ADMISSION DATE**
11/30/2025

**SECTION 14: ADMIT TYPE**
1 – Emergency

**SECTION 17: PATIENT DISCHARGE STATUS**
01 – Discharged to Home / Self-Care

**SECTION 18–28 CONDITION CODES**
05 Lien
39 Private Room Medically Necessary

---

**BILLING LINES:**

| 42 REV CD | 43 DESCRIPTION | 45 SERVICE DATE | 46 UNITS | 47 TOTAL CHARGES | 48 NON-COV CHARGES | 49 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 0120 | Room & Board – Semi-Private (3 days) | 11/30/2025 | 3 | $8,750.00 | $0.00 | |
| 0490 | Psychotherapy Session, 60 min | 12/15/2025 | 1 | $250.00 | $0.00 | |
| 0490 | Electrocardiogram, routine ECG with at least 12 leads | 12/15/2025 | 1 | $95.00 | $0.00 | |

---

**PAGE TOTALS**
$4,095.00

*(Note: The sum of the listed line items is $9,095.00, which conflicts with the displayed page total.)*

**SECTION 50: PAYER NAME**
MediShield Health Ins. Ltd.

**SECTION 55: ESTAMTED AMT DUE**
$4,095.00

**SECTION 56: BILLING NPI**
NPI-5610293847

**SECTION 57: GROUP NUMBER**
GRP-89131

**SECTION 58: INSURED'S NAME**
Christopher Harris I.

**SECTION 60: INSURED UNIQUE ID**
MED-GLB-6770619

**SECTION 67: PRINCIPAL DIAGNOSIS (ICD-10-CM)**
I25.10 – Coronary artery disease without angina pectoris

**SECTION 69: ADMITTING DIAGNOSIS**
G43.909 – Migraine, unspecified, not intractable

**SECTION 74: PRINCIPAL PROCEDURE (CPT)**
CPT 90837 – Psychotherapy Session, 60 min
Procedure Date: 12/15/2025

**SECTION 76: ATTENDING PHYSICIAN**
Dr. Rachel Kim, MD
NPI: NPI-5610293847

**Attending Physician Signature**
Dr. Rachel Kim, MD
Date: 12/03/2025

**Bottom Barcode Label**
UB04-CIM-2024-00001
"""

In [28]:
print(SAMPLE_CONTEXT_TYPE2)

Based on the provided image of the Health Insurance Claim Form (CMS-1500), here is the extracted text and key data:

**Form Header:**
*   **Title:** HEALTH INSURANCE CLAIM FORM
*   **Format:** CMS-1500 (02-12) / UB-04 Format
*   **Approval:** APPROVED BY NATIONAL UNIFORM CLAIM COMMITTEE

**Patient and Insurance Information:**
*   **1. INSURANCE TYPE:** ☑ GROUP HEALTH PLAN
*   **1a. INSURED'S I.D. NUMBER (policy):** MED-GLD-4567281
*   **2. PATIENT'S NAME (Last, First, MI):** Patricia Miller H.
*   **3. PATIENT'S BIRTH DATE:** 01/26/1997
*   **3. SEX:** F
*   **5. PATIENT'S ADDRESS:** 654 Elm St., Rochester, MN
*   **6. PATIENT RELATIONSHIP:** ☑ Self
*   **4. INSURED'S NAME:** SAME AS PATIENT
*   **7. INSURED'S ADDRESS:** SAME AS PATIENT
*   **8. RESERVED FOR NUCC USE:** ☐ Other
*   **11. INSURED'S GROUP NO.:** GRP-95909
*   **11a. INSURED'S DATE OF BIRTH / SEX:** 01/26/1997 | SEX: 
*   **11d. IS THERE ANYOTHER HEALTH BENEFIT PLAN?** ☐ YES ☐ NO
*   **13. INSURED'S OR AUTHORIZED PERSON'S

In [ ]:
# Cell 3
import json

result = extract_claim(SAMPLE_CONTEXT, model=MODEL_NAME)

print(json.dumps(result, indent=2))

In [33]:
# Cell 2
def extract_claim(
    context: str,
    model: str = "llama3.1",
    base_url: str = "http://localhost:11434",
    temperature: float = 0.0,
    cpt_description_lookup: dict | None = None,
) -> dict:
    """
    Extract structured claim data (with CPT code+description) from a UB-04
    or CMS-1500 claim passed in as raw text.

    Returns:
      {
        "extracted_fields": {
            "claim_amount": float | None,
            "icd10_codes": [str, ...],
            "cpt_codes": [{"code": str, "description": str | None}, ...],
            "provider_npi": str | None,
            "service_date": "YYYY-MM-DD" | None,
            "physician_signature": str | None,
            "confidence": float | None,
            "notes": str | None,
        },
        "schema_valid": bool,
        "validation_errors": [{"field": str, "code": str, "message": str}, ...],
      }
    """
    # ---------------- lazy imports (keeps notebook namespace clean) ---------
    import re
    import json
    from typing import Optional, List, Dict, Any, Union

    from pydantic import BaseModel, Field
    from langchain_ollama import ChatOllama
    from langchain_core.runnables import RunnableLambda
    from langchain_core.prompts import ChatPromptTemplate

    # ---------------- schema ------------------------------------------------
    class CPTCode(BaseModel):
        code: str = Field(..., description="5-digit CPT code, e.g. 99213.")
        description: Optional[str] = Field(
            default=None,
            description="Plain-English procedure description.",
        )

    class ClaimExtraction(BaseModel):
        claim_amount: Optional[float] = Field(
            default=None, description="Total billed/claim amount in USD."
        )
        icd10_codes: List[str] = Field(
            default_factory=list, description="ICD-10 diagnosis codes."
        )
        cpt_codes: List[Union[str, CPTCode]] = Field(
            default_factory=list,
            description='CPT codes as strings or {"code","description"} objects.',
        )
        provider_npi: Optional[str] = Field(
            default=None, description="10-digit NPI (billing or rendering provider)."
        )
        service_date: Optional[str] = Field(
            default=None, description="Primary date of service, YYYY-MM-DD."
        )
        physician_signature: Optional[str] = Field(
            default=None, description="Attending/rendering physician signature."
        )
        confidence: Optional[float] = Field(
            default=None, description="Model confidence 0.0-1.0."
        )
        notes: Optional[str] = Field(
            default=None, description="Ambiguity / missing-field notes."
        )

    # ---------------- prompts -----------------------------------------------
    # Every literal { or } inside the prompt must be escaped as {{ / }}
    # because ChatPromptTemplate treats {x} as a template variable.
    # Only {context} stays single-braced.
    system_prompt = """You are a meticulous medical claims extraction agent.
You may be given either a UB-04 (CMS-1450) claim or a CMS-1500 claim as text.

Identify the form type first:
- If the text mentions "UB-04", "CMS-1450", "TYPE OF BILL", "REV CD", "PRINCIPAL
  PROCEDURE", or shows revenue codes (4-digit like 0120, 0490), it is a UB-04.
- If the text mentions "CMS-1500", "SERVICES / PROCEDURES", "PLACE OF SERVICE",
  or lists CPT/HCPCS per row, it is a CMS-1500.

Then extract these fields:

1. claim_amount
   - UB-04: use PAGE TOTALS or SECTION 55 ESTIMATED AMT DUE.
   - CMS-1500: use TOTAL CHARGE (box 28).
   - Numeric only (strip $ and commas).

2. icd10_codes
   - All ICD-10-CM codes present (UB-04 sections 67/69/72; CMS-1500 box 21).
   - Uppercase, keep the dot (e.g. "i2510" -> "I25.10").
   - Example: I25.10, G43.909, K35.80.

3. cpt_codes -> list of OBJECTS, one per procedure:
   {{"code": "<5-digit CPT>", "description": "<plain-English description>"}}
   - UB-04 sources:
       * SECTION 74 PRINCIPAL PROCEDURE (CPT 90837 - Psychotherapy Session, 60 min)
       * BILLING LINES whose description contains a procedure code
       * Any "CPT <digits>" mentions
     Revenue codes (0120, 0490) are NOT CPT codes - do not emit them as CPT.
     If a revenue code line also names a CPT (e.g. "0490 | ... | 90837"), emit
     only the CPT (90837) with its description.
   - CMS-1500 sources:
       * Section 24 rows -> CPT/HCPCS column (e.g. 59400, 27447) with the
         DESCRIPTION shown in that row.
   - description = the text next to the code in the source. Truncate at the
     first line break or "|". If no description is present, use null.
   - Do NOT invent descriptions. Do NOT include ICD-10 codes here.
   - De-duplicate codes, keeping the first description seen.

4. provider_npi
   - UB-04: SECTION 56 BILLING NPI, or attending-physician NPI (section 76).
   - CMS-1500: box 33 billing NPI or box 24J rendering NPI.
   - Normalize to 10 digits (strip non-digits; drop "NPI-" prefix).
   - If multiple, prefer the BILLING provider NPI.

5. service_date
   - CMS-1500: earliest DATE OF SERVICE in section 24.
   - UB-04: SECTION 6 STATEMENT COVERS FROM (start of period), else earliest
     line service date.
   - Normalize to YYYY-MM-DD.

6. physician_signature
   - UB-04: Attending Physician Signature name (e.g. "Dr. Rachel Kim, MD").
   - CMS-1500: box 31 SIGNATURE OF PHYSICIAN OR SUPPLIER name
     (e.g. "Dr. Brian Wallace, MD").

Rules:
- Only extract what is explicitly present. Do NOT hallucinate.
- Missing scalar -> null; missing list -> [].
- Output MUST be valid JSON matching the schema. No markdown, no commentary.
"""

    user_prompt = """Extract the claim fields from the context below.

--- CONTEXT START ---
{context}
--- CONTEXT END ---

Return ONLY valid JSON with these keys:
claim_amount, icd10_codes, cpt_codes, provider_npi, service_date,
physician_signature, confidence, notes

cpt_codes MUST be a list of objects:
[{{"code": "85025", "description": "CBC with differential"}}, ...]
"""

    # ---------------- regexes / constants -----------------------------------
    ICD10_RE = re.compile(r"^[A-Z]\d{2}(\.\d{1,4})?$")
    CPT_RE   = re.compile(r"^\d{5}$")
    NPI_RE   = re.compile(r"^\d{10}$")
    DATE_RE  = re.compile(r"^\d{4}-\d{2}-\d{2}$")

    REQUIRED_FIELDS = {
        "claim_amount":        "claim_amount is required and cannot be null.",
        "icd10_codes":         "icd10_codes is required and cannot be empty.",
        "cpt_codes":           "cpt_codes is required and cannot be empty.",
        "provider_npi":        "provider_npi is required and cannot be null.",
        "service_date":        "service_date is required and cannot be null.",
        "physician_signature": "physician_signature is required and cannot be null.",
    }

    lookup: Dict[str, str] = dict(cpt_description_lookup or {})

    # ---------------- helpers -----------------------------------------------
    def normalize_date(raw):
        if not raw:
            return None
        raw = str(raw).strip()
        if DATE_RE.match(raw):
            return raw
        m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{4})$", raw)
        if m:
            mm, dd, yyyy = m.groups()
            return f"{yyyy}-{int(mm):02d}-{int(dd):02d}"
        m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})$", raw)
        if m:
            a, b, yyyy = m.groups()
            if len(yyyy) == 2:
                yyyy = "20" + yyyy
            return f"{yyyy}-{int(a):02d}-{int(b):02d}"
        return None

    def normalize_npi(raw):
        if raw is None:
            return None
        digits = re.sub(r"\D", "", str(raw))
        return digits if NPI_RE.fullmatch(digits) else None

    def normalize_cpt_entry(c):
        if isinstance(c, str):
            return {"code": c.strip(), "description": None}
        if isinstance(c, dict):
            code = str(c.get("code", "")).strip()
            raw_desc = c.get("description")
            desc = str(raw_desc).strip() if raw_desc is not None else None
            return {"code": code, "description": desc or None}
        return None

    def validate_and_format(result):
        errors: List[Dict[str, str]] = []
        out: Dict[str, Any] = {}

        # claim_amount
        amt = result.get("claim_amount")
        if amt is None:
            out["claim_amount"] = None
        else:
            try:
                if isinstance(amt, str):
                    amt = amt.replace(",", "").replace("$", "").strip()
                out["claim_amount"] = float(amt)
            except (TypeError, ValueError):
                out["claim_amount"] = None
                errors.append({
                    "field": "claim_amount",
                    "code": "INVALID_FORMAT",
                    "message": f"claim_amount must be numeric, got: {result.get('claim_amount')!r}",
                })

        # icd10_codes
        icd_clean = []
        for c in (result.get("icd10_codes") or []):
            if not isinstance(c, str):
                errors.append({
                    "field": "icd10_codes",
                    "code": "INVALID_FORMAT",
                    "message": f"ICD-10 code must be a string, got: {c!r}",
                })
                continue
            cu = c.upper().strip()
            if ICD10_RE.fullmatch(cu):
                icd_clean.append(cu)
            else:
                errors.append({
                    "field": "icd10_codes",
                    "code": "INVALID_FORMAT",
                    "message": f"Invalid ICD-10 code format: {c!r}",
                })
        out["icd10_codes"] = icd_clean

        # cpt_codes -> canonical [{"code","description"}, ...]
        cpt_clean = []
        seen_codes = set()
        for c in (result.get("cpt_codes") or []):
            entry = normalize_cpt_entry(c)
            if entry is None:
                errors.append({
                    "field": "cpt_codes",
                    "code": "INVALID_FORMAT",
                    "message": f"CPT entry must be string or object, got: {c!r}",
                })
                continue
            code, desc = entry["code"], entry["description"]
            if not CPT_RE.fullmatch(code):
                errors.append({
                    "field": "cpt_codes",
                    "code": "INVALID_FORMAT",
                    "message": f"Invalid CPT code format: {code!r} (from {c!r})",
                })
                continue
            if desc is None:
                desc = lookup.get(code)
            if code in seen_codes:
                continue
            seen_codes.add(code)
            cpt_clean.append({"code": code, "description": desc})
        out["cpt_codes"] = cpt_clean

        # provider_npi
        npi_norm = normalize_npi(result.get("provider_npi"))
        out["provider_npi"] = npi_norm
        if result.get("provider_npi") not in (None, "") and npi_norm is None:
            errors.append({
                "field": "provider_npi",
                "code": "INVALID_FORMAT",
                "message": f"Invalid NPI (must be 10 digits): {result.get('provider_npi')!r}",
            })

        # service_date
        raw_date = result.get("service_date")
        norm = normalize_date(raw_date) if raw_date else None
        if raw_date and not norm:
            errors.append({
                "field": "service_date",
                "code": "INVALID_FORMAT",
                "message": f"Could not parse service_date: {raw_date!r}",
            })
        out["service_date"] = norm

        # physician_signature
        sig = result.get("physician_signature")
        out["physician_signature"] = (
            str(sig).strip() if sig is not None and str(sig).strip() else None
        )

        # confidence / notes
        conf = result.get("confidence")
        try:
            out["confidence"] = float(conf) if conf is not None else None
        except (TypeError, ValueError):
            out["confidence"] = None
        notes = result.get("notes")
        out["notes"] = str(notes).strip() if notes else None

        # required-field checks
        for field, msg in REQUIRED_FIELDS.items():
            val = out.get(field)
            is_missing = val is None or (isinstance(val, list) and len(val) == 0)
            if is_missing and not any(e["field"] == field for e in errors):
                errors.append({
                    "field": field,
                    "code": "REQUIRED_MISSING",
                    "message": msg,
                })

        return {
            "extracted_fields": out,
            "schema_valid": len(errors) == 0,
            "validation_errors": errors,
        }

    def safe_parse(message):
        raw = message.content if hasattr(message, "content") else str(message)
        raw = (
            raw.strip()
            .removeprefix("```json")
            .removeprefix("```")
            .removesuffix("```")
            .strip()
        )
        try:
            data = json.loads(raw)
        except json.JSONDecodeError as e:
            return {
                "extracted_fields": {
                    "claim_amount": None,
                    "icd10_codes": [],
                    "cpt_codes": [],
                    "provider_npi": None,
                    "service_date": None,
                    "physician_signature": None,
                },
                "schema_valid": False,
                "validation_errors": [{
                    "field": "_raw_output",
                    "code": "INVALID_JSON",
                    "message": f"Model did not return valid JSON: {e}",
                }],
            }
        try:
            validated = ClaimExtraction(**data)
            return validate_and_format(validated.model_dump())
        except Exception as e:
            return {
                "extracted_fields": data,
                "schema_valid": False,
                "validation_errors": [{
                    "field": "_schema",
                    "code": "VALIDATION_ERROR",
                    "message": str(e),
                }],
            }

    # ---------------- run ----------------------------------------------------
    if not context or not context.strip():
        raise ValueError("Context is empty.")

    llm = ChatOllama(
        base_url=base_url,
        model=model,
        temperature=temperature,
        format="json",
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", user_prompt),
    ])
    chain = prompt | llm | RunnableLambda(safe_parse)
    return chain.invoke({"context": context})

In [39]:
cms400_result = extract_claim(SAMPLE_CONTEXT_TYPE2, model=OLLAMA_MODEL)

In [ ]:
cms1500_result = ""

In [42]:
cms1500_result

{'extracted_fields': {'claim_amount': '4095',
  'icd10_codes': ['I25.10', 'G43.909'],
  'cpt_codes': [{'code': '90837',
    'description': 'Psychotherapy Session, 60 min'},
   {'code': '90837', 'description': 'Psychotherapy Session, 60 min'}],
  'provider_npi': '5610293847',
  'service_date': '2025-11-30',
  'physician_signature': 'Dr. Rachel Kim, MD',
  'confidence': 'high',
  'notes': 'The sum of the listed line items is $9,095.00, which conflicts with the displayed page total of $4,095.00.'},
 'schema_valid': False,
 'validation_errors': [{'field': '_schema',
   'code': 'VALIDATION_ERROR',
   'message': "1 validation error for ClaimExtraction\nconfidence\n  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='high', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.14/v/float_parsing"}]}

In [40]:
cms400_result

{'extracted_fields': {'claim_amount': 18900,
  'icd10_codes': ['K35.80'],
  'cpt_codes': [{'code': '59400',
    'description': 'Routine obstetric care including prenatal care, labor and delivery, and postpartum care'},
   {'code': '27447',
    'description': 'Arthroplasty, knee, condyle and patella'}],
  'provider_npi': '8374651029',
  'service_date': '2025-11-15',
  'physician_signature': 'Dr. Brian Wallace, MD',
  'confidence': 'high',
  'notes': 'The procedure descriptions are truncated in the image. The diagnosis code K35.80 for appendicitis appears inconsistent with the listed CPT codes 59400 (obstetric care) and 27447 (knee arthroplasty). This is likely an error in the sample form data.'},
 'schema_valid': False,
 'validation_errors': [{'field': '_schema',
   'code': 'VALIDATION_ERROR',
   'message': "1 validation error for ClaimExtraction\nconfidence\n  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='high', input_type=str]\n  